# device-consistent-construct — worked example 1: Build a multiplicative all-ones mask matching the input's device + dtype

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `device-consistent-construct`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A helper tensor allocated inside `forward()` must live on the **same device and dtype** as the operand it interacts with. `t.ones(shape, device=x.device, dtype=x.dtype)` threads both attributes straight from `x`, so the elementwise product never triggers a device-mismatch error or a silent dtype upcast. The naive `t.ones(shape).to(x.device)` allocates on CPU first and defaults to `float32`, which corrupts a `bfloat16`/`float64` input.

## Worked solution

**Goal:** a Module that multiplies `x` by a freshly built all-ones tensor (a deliberate no-op product, used to audit the allocation).

**Step 1 — read device + dtype from the operand.** Inside `forward`, `x.device` and `x.dtype` describe exactly where the result must end up. We do *not* hardcode `'cpu'` or `t.float32` — those would break the moment `x` is a GPU `bfloat16` tensor.

**Step 2 — allocate the helper with both kwargs.** `mask = t.ones(x.shape, device=x.device, dtype=x.dtype)`. Passing `device=` means the buffer is created directly on the target device (no host allocation + copy). Passing `dtype=` means the buffer already matches `x`'s dtype, so the multiply below stays in that dtype.

**Step 3 — combine.** `mask * x` is mathematically `x`, but the product forces PyTorch to align the two tensors. If `mask` were on the wrong device this line raises `RuntimeError`; if it were the wrong dtype the result would upcast. Because we threaded both kwargs, the output dtype equals `x.dtype` exactly.

**Why it works:** the construction kwargs make the helper *born* consistent with `x`, so no later `.to()` correction or implicit promotion is needed.

In [ ]:
def make_masked_identity():
    class MaskedIdentity(t.nn.Module):
        def forward(self, x):
            mask = t.ones(x.shape, device=x.device, dtype=x.dtype)
            return mask * x
    return MaskedIdentity()

mod = make_masked_identity()
for dt in (t.float32, t.float64, t.bfloat16):
    x = t.randn(2, 3).to(dt)
    y = mod(x)
    print(dt, y.dtype, y.device, t.allclose(y.float(), x.float()))